# 03. Temporal validation и метрики

Разделяем прошлое и будущее небольшими шагами, строим ground truth и сохраняем границы трёх окон.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Mounted at /content/drive
Корень проекта: /content/drive/MyDrive/fashion-recommender-system


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [ ]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем pandas, метрики и JSON persistence  
**Зачем:** в notebook не нужны модели  
**Что получим:** минимальный набор импортов

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from fashion_recommender.data import load_transactions
from fashion_recommender.evaluation import (
    average_precision_at_k, candidate_recall_at_k, hit_rate_at_k,
    map_at_k, mean_recall_at_k, recall_at_k,
)
from fashion_recommender.persistence import save_json

### Пути и параметры

**Что делаем:** задаём каталоги и длину target-периода  
**Зачем:** границы должны использоваться одинаково дальше  
**Что получим:** `FUTURE_DAYS = 7` и reproducible seed

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FUTURE_DAYS = 7
RANDOM_STATE = 42

### Загрузка транзакций

**Что делаем:** читаем историю покупок  
**Зачем:** cutoff определяется по фактической последней дате  
**Что получим:** DataFrame с корректным datetime

In [ ]:
transactions = load_transactions(TRANSACTIONS_PATH)
print("Форма transactions:", transactions.shape)
display(transactions.head())

Форма transactions: (1048575, 5)


,t_dat,customer_id,article_id,price,sales_channel_id
0,2019-11-05,3e2b60b679e62fb49516105b975560082922011dd752ec...,0698328010,0.016932,2
1,2019-05-22,89647ac2274f54c770aaa4b326e0eea09610c252381f37...,0760597002,0.033881,2
2,2019-05-10,2ebe392150feb60ca89caa8eff6c08b7ef1138cd6fdc71...,0488561032,0.016932,2
3,2019-08-26,7b3205de4ca17a339624eb5e3086698e9984eba6b47c56...,0682771001,0.033881,2
4,2019-08-10,3b77905de8b32045f08cedb79200cdfa477e9562429a39...,0742400033,0.003220,1


### Максимальная дата

**Что делаем:** находим последнюю дату данных  
**Зачем:** от неё отсчитывается последняя неделя  
**Что получим:** `last_date`

In [ ]:
last_date = transactions["t_dat"].max()
print("Последняя дата:", last_date)

Последняя дата: 2019-12-31 00:00:00


### Начало target-периода

**Что делаем:** вычисляем cutoff для семи календарных дней  
**Зачем:** обе границы недели включительны  
**Что получим:** `cutoff_date`

In [ ]:
cutoff_date = last_date - pd.Timedelta(days=FUTURE_DAYS - 1)
print("Target начинается:", cutoff_date)

Target начинается: 2019-12-25 00:00:00


### History

**Что делаем:** берём строки строго раньше cutoff  
**Зачем:** модели не должны видеть target-неделю  
**Что получим:** таблицу прошлого

In [ ]:
history = transactions[
    transactions["t_dat"] < cutoff_date
].copy()
print("History:", history.shape)

History: (1035285, 5)


### Future

**Что делаем:** берём строки начиная с cutoff  
**Зачем:** это ответы для offline-оценки  
**Что получим:** таблицу будущего

In [ ]:
future = transactions[
    transactions["t_dat"] >= cutoff_date
].copy()
print("Future:", future.shape)

Future: (13290, 5)


### Проверка диапазонов

**Что делаем:** сравниваем крайние даты и число строк  
**Зачем:** так обнаруживается пересечение или потеря данных  
**Что получим:** явные leakage-checks

In [ ]:
print("History заканчивается:", history["t_dat"].max())
print("Future начинается:", future["t_dat"].min())
print("Строк после split:", len(history) + len(future))

assert history["t_dat"].max() < future["t_dat"].min()
assert len(history) + len(future) == len(transactions)

History заканчивается: 2019-12-24 00:00:00
Future начинается: 2019-12-25 00:00:00
Строк после split: 1048575


### Пользователи с историей

**Что делаем:** собираем ID из history  
**Зачем:** персональные модели работают только для известных клиентов  
**Что получим:** множество `history_users`

In [ ]:
history_users = set(history["customer_id"])
print("Пользователей в history:", len(history_users))

Пользователей в history: 454441


### Cold start

**Что делаем:** находим future-пользователей без history  
**Зачем:** их нельзя смешивать с обычной персональной оценкой  
**Что получим:** отдельный список cold-start ID

In [ ]:
future_users = set(future["customer_id"])
cold_start_users = future_users - history_users
print("Пользователей в future:", len(future_users))
print("Cold-start пользователей:", len(cold_start_users))

Пользователей в future: 11384
Cold-start пользователей: 3794


### Evaluation future

**Что делаем:** оставляем future только известных пользователей  
**Зачем:** ground truth должен соответствовать доступной истории  
**Что получим:** `future_evaluation`

In [ ]:
future_evaluation = future[
    future["customer_id"].isin(history_users)
].copy()
print("Evaluation future:", future_evaluation.shape)

Evaluation future: (9007, 5)


### Уникальные будущие пары

**Что делаем:** сортируем и удаляем повторные user-item покупки  
**Зачем:** один товар учитывается в метриках один раз  
**Что получим:** `future_unique`

In [ ]:
future_unique = (
    future_evaluation
    .sort_values("t_dat")
    .drop_duplicates(["customer_id", "article_id"])
)
display(future_unique.head())

,t_dat,customer_id,article_id,price,sales_channel_id
782025,2019-12-25,0e2ae0450bd6ea975916e874b56ca3825d17939b108371...,0719295001,0.016932,2
216603,2019-12-25,e7c0d1d1d130018afc0bac18a50dcc8357b957ba8d4b21...,0822201002,0.101678,2
432391,2019-12-25,a4a98e3cc5c34396e2ede8921553aad7d45f89b4c3e343...,0573085028,0.032797,2
244973,2019-12-25,1a8ff5bca9a06d93488b597eaba364f4d96d936a3c96b6...,0578113001,0.030271,2
637251,2019-12-25,a4714776e4465bc2dad6012862655da96d2a3862d0503b...,0777504001,0.025407,2


### Ground truth

**Что делаем:** собираем список будущих товаров каждого пользователя  
**Зачем:** метрики сравнивают рекомендации с этим словарём  
**Что получим:** полный `ground_truth` без искусственного обрезания

In [ ]:
ground_truth = (
    future_unique
    .groupby("customer_id", sort=False)["article_id"]
    .apply(list)
    .to_dict()
)
print("Пользователей в ground truth:", len(ground_truth))

Пользователей в ground truth: 7590


### Пример пользователя

**Что делаем:** показываем один будущий список  
**Зачем:** проверяем смысл структуры до расчёта метрик  
**Что получим:** один customer ID и его товары

In [ ]:
example_customer = next(iter(ground_truth))
print("Пользователь:", example_customer)
print("Future items:", ground_truth[example_customer])

Пользователь: 0e2ae0450bd6ea975916e874b56ca3825d17939b10837140df8af6b271662ae6
Future items: ['0719295001', '0764582001', '0764520002', '0650279001', '0703474001', '0761622001']


### Маленький пример

**Что делаем:** задаём actual и recommended items вручную  
**Зачем:** формулы проще проверить на четырёх позициях  
**Что получим:** данные для ручного Recall и AP

In [ ]:
actual_items = ["A", "B", "C"]
recommended_items = ["A", "X", "B", "Y"]
K = 4

actual_set = set(actual_items)
top_k = list(dict.fromkeys(recommended_items))[:K]
print("Actual:", actual_items)
print("Top-K:", top_k)

Actual: ['A', 'B', 'C']
Top-K: ['A', 'X', 'B', 'Y']


### Recall вручную

**Что делаем:** делим число найденных товаров на число actual items  
**Зачем:** Recall отвечает за полноту  
**Что получим:** ручное значение Recall@4

In [ ]:
hits = actual_set & set(top_k)
manual_recall = len(hits) / len(actual_set)
print("Попадания:", hits)
print("Recall@4:", manual_recall)

Попадания: {'A', 'B'}
Recall@4: 0.6666666666666666


### AP вручную

**Что делаем:** суммируем precision только на позициях попаданий  
**Зачем:** AP учитывает порядок рекомендаций  
**Что получим:** ручное значение AP@4

In [ ]:
precision_sum = 0.0
hits_so_far = 0
for rank, article_id in enumerate(top_k, start=1):
    if article_id in actual_set:
        hits_so_far += 1
        precision_sum += hits_so_far / rank

manual_ap = precision_sum / min(len(actual_set), K)
print("AP@4:", manual_ap)

AP@4: 0.5555555555555555


### Проверка готовыми метриками

**Что делаем:** сравниваем ручные значения с функциями из src  
**Зачем:** единые функции нужны во всех следующих notebooks  
**Что получим:** совпадающие Recall и AP

In [ ]:
print("Recall function:", recall_at_k(actual_items, recommended_items, K))
print("AP function:", average_precision_at_k(actual_items, recommended_items, K))

assert np.isclose(manual_recall, recall_at_k(actual_items, recommended_items, K))
assert np.isclose(manual_ap, average_precision_at_k(actual_items, recommended_items, K))

Recall function: 0.6666666666666666
AP function: 0.5555555555555555


### Групповые метрики

**Что делаем:** считаем Recall, MAP, HitRate и Candidate Recall на двух пользователях  
**Зачем:** видим различие метрик до экспериментов  
**Что получим:** четыре проверочных значения

In [ ]:
example_truth = {"u1": ["A", "B"], "u2": ["C"]}
example_recommendations = {"u1": ["A", "X"], "u2": ["Y", "C"]}

print("Mean Recall@2:", mean_recall_at_k(example_truth, example_recommendations, 2))
print("MAP@2:", map_at_k(example_truth, example_recommendations, 2))
print("HitRate@2:", hit_rate_at_k(example_truth, example_recommendations, 2))
print("Candidate Recall@2:", candidate_recall_at_k(example_truth, example_recommendations, 2))

Mean Recall@2: 0.75
MAP@2: 0.5
HitRate@2: 1.0
Candidate Recall@2: 0.75


### Границы трёх окон

**Что делаем:** задаём train, validation и test явно  
**Зачем:** следующие notebooks используют одинаковый protocol  
**Что получим:** шесть дат без цикла по pipeline

In [ ]:
train_cutoff = last_date - pd.Timedelta(days=20)
train_end = train_cutoff + pd.Timedelta(days=6)
validation_cutoff = last_date - pd.Timedelta(days=13)
validation_end = validation_cutoff + pd.Timedelta(days=6)
test_cutoff = last_date - pd.Timedelta(days=6)
test_end = last_date

print("Train:", train_cutoff.date(), "—", train_end.date())
print("Validation:", validation_cutoff.date(), "—", validation_end.date())
print("Test:", test_cutoff.date(), "—", test_end.date())

Train: 2019-12-11 — 2019-12-17
Validation: 2019-12-18 — 2019-12-24
Test: 2019-12-25 — 2019-12-31


### Конфигурация окон

**Что делаем:** собираем даты в компактный JSON-словарь  
**Зачем:** следующие этапы не должны вычислять границы заново  
**Что получим:** `window_config`

In [ ]:
window_config = {
    "train": {"cutoff_date": train_cutoff, "target_end_date": train_end},
    "validation": {
        "cutoff_date": validation_cutoff,
        "target_end_date": validation_end,
    },
    "test": {"cutoff_date": test_cutoff, "target_end_date": test_end},
}
display(pd.DataFrame(window_config).T)

,cutoff_date,target_end_date
train,2019-12-11,2019-12-17
validation,2019-12-18,2019-12-24
test,2019-12-25,2019-12-31


### Сохранение окон

**Что делаем:** записываем только даты  
**Зачем:** notebooks 04–08 загрузят один общий контракт  
**Что получим:** `temporal_windows.json`

In [ ]:
windows_path = PROCESSED_DIR / "temporal_windows.json"
save_json(window_config, windows_path)
print("Сохранено:", windows_path)

Сохранено: /content/drive/MyDrive/fashion-recommender-system/data/processed/temporal_windows.json
